# N-ATLaS DevKit — Colab demo

Run Nigeria's open language model **N-ATLaS** on a free Colab GPU, then use it three ways:
the **Python SDK**, an **OpenAI-compatible API**, and a **web playground**.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU*.
2. Sign in at huggingface.co and accept the terms on the [NCAIR1/N-ATLaS](https://huggingface.co/NCAIR1/N-ATLaS) model page.
3. Create a *read* token at huggingface.co/settings/tokens and add it in Colab's **🔑 Secrets** panel as `HF_TOKEN`.

*N-ATLaS is an initiative of the Federal Ministry of Communications, Innovation and Digital Economy, and powered by Awarri Technologies.*

## 1. Install

In [ ]:
!pip install -q "natlas-devkit[all] @ git+https://github.com/codeunitdev/natlas-devkit.git" openai

## 2. Hugging Face login

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

## 3. Load N-ATLaS in 4-bit (about 6 GB of GPU memory; first download takes a few minutes)

In [ ]:
from natlas import NAtlas
llm = NAtlas(quantization="4bit")
print("Loaded", llm.model_id)

## 4. Python SDK — one model, four languages

In [ ]:
prompts = {
    "Hausa":   "menene ake nufi da gwagwarmaya",
    "Yoruba":  "Báwo ni mo ṣe lè dáàbò bo ara mi lọ́wọ́ ibà?",
    "Igbo":    "Kedu ka m ga-esi chebe onwe m pụọ n'ọrịa ịba?",
    "English": "Explain in simple terms how a farmer can save money with a cooperative.",
}
for lang, q in prompts.items():
    print(f"=== {lang} ===\nQ: {q}\nA: {llm.ask(q, max_new_tokens=250)}\n")

### Translation and forced reply language

In [ ]:
print(llm.translate("Wash your hands with soap before eating.", to="Hausa"))
print(llm.ask("What are the early signs of malaria?", language="Igbo", max_new_tokens=200))

### Streaming

In [ ]:
for piece in llm.stream_chat([{"role": "user", "content": "Give me three tips for saving money as a student."}], max_new_tokens=250):
    print(piece, end="", flush=True)

## 5. OpenAI-compatible API

Start the API server in the background (reusing the loaded model), then call it with the **official OpenAI client**.
Any app built for OpenAI switches to N-ATLaS by changing `base_url`.

In [ ]:
import threading, time, uvicorn
from natlas.server import create_app

server = uvicorn.Server(uvicorn.Config(create_app(engine=llm), host="127.0.0.1", port=8000, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

from openai import OpenAI
client = OpenAI(base_url="http://127.0.0.1:8000/v1", api_key="local")
resp = client.chat.completions.create(
    model="n-atlas",
    messages=[{"role": "user", "content": "menene ake nufi da gwagwarmaya"}],
    max_tokens=200,
)
print(resp.choices[0].message.content)
print(resp.usage)

## 6. Web playground (opens a temporary public link)

In [ ]:
from natlas.playground import build_demo
build_demo(engine=llm).queue().launch(share=True)